# Notebook 20 — Fold-safe normalization and staging pilot (ISLES2024)

**Purpose:** validate the *actual executable* candidate transforms and the architecture of separate per-fold nnU-Net workspaces for **Dataset506 (NCCT)** and **Dataset507 (NCCT + 4 perfusion maps)**, without starting any training.

**Runtime:** CPU, Google Drive mounted. The candidate anatomical mask is a *head-tissue proxy*, **not a validated clinical brain mask**. The pilot uses **6 training patients** from fold 2 and **one held-out patient** (`sub_stroke0016`) for QC, **NOT the full 119-patient training fold**.

This experiment does **not** establish clinical validity or authorize the 506/507 full five-fold experiments. All original Dataset504/505 files and models remain unchanged. **No GPU is used.**

## Leakage-safe workspace design

Each full fold will use distinct `nnUNet_raw`, `nnUNet_preprocessed`, and `nnUNet_results` workspaces. Calibration may read **only training patient images**; holdout cases enter solely after parameters are locked. Training images are staged into `imagesTr`, heldout images into `imagesTs`, and **only training labels** into `labelsTr`. Both arms use **identical pre-transformed NCCT bytes**, and set all channel names to nnU-Net's recognized `noNorm` mapping to prevent a second standard normalization stage.

**Still blocked before model training:** anatomical-mask clinical QC, the prospective bounds/cap policy, identical 3D full-resolution architecture and spacing plans, full-fold training-only recalibration, inference parity, and post-resampling quality checks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Verify the frozen input dataset and source scripts

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW = ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLIT = ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'/'splits_final.json'
REPORTS = ROOT/'results'/'wu20_fold_isolated_pilot'
REPO_COMMIT = '9ed420a6b6db7a2621ad831766afdd671e5c427b'
BASE = f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{REPO_COMMIT}'
SOURCE = Path('/content/wu20_source')
PKG = SOURCE/'src'/'hybridstrokeseg'
PKG.mkdir(parents=True,exist_ok=True)
(SOURCE/'scripts').mkdir(parents=True, exist_ok=True)

def sha256(path):
    with Path(path).open('rb') as fp:
        h=hashlib.sha256()
        for block in iter(lambda:fp.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

assert RAW.is_dir(), f'Missing Dataset505 staged raw images: {RAW}'
assert SPLIT.is_file(), f'Missing frozen patient split: {SPLIT}'
assert sha256(SPLIT) == '664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
for channel in range(5):
    path=RAW/'imagesTr'/f'sub_stroke0016_{channel:04d}.nii.gz'
    assert path.is_file(), f'Missing case for fold-2 pilot: {path}'
assert (RAW/'labelsTr'/'sub_stroke0016.nii.gz').is_file()
files = {
    'src/hybridstrokeseg/wu19_normalization_protocol.py': PKG/'wu19_normalization_protocol.py',
    'src/hybridstrokeseg/wu20_fold_preprocessing.py': PKG/'wu20_fold_preprocessing.py',
    'scripts/stage_isles2024_fold_pilot.py': SOURCE/'scripts'/'stage_isles2024_fold_pilot.py',
}
for relative,dest in files.items():
    subprocess.run(['curl','-fLsS','--retry','5','--retry-delay','3',f'{BASE}/{relative}', '-o',str(dest)],check=True)
    assert dest.stat().st_size > 500, f'Truncated pinned source: {relative}'
    print('Source', relative, dest.stat().st_size, 'bytes', sha256(dest)[:14])
print('Frozen split SHA256:',sha256(SPLIT))
print('Candidate WU20 pinned commit:',REPO_COMMIT)


## 2. CPU dependencies and small synthetic smoke test

Only image I/O, NumPy, and SciPy are needed. The model training environment is **not** installed or launched.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--only-binary=:all:',
                'numpy>=1.26,<3', 'scipy>=1.12,<2', 'nibabel>=5.2,<6'],check=True)
sys.path.insert(0,str(SOURCE/'src'))
from hybridstrokeseg.wu20_fold_preprocessing import ncct_head_tissue_support
import numpy as np
synthetic = np.full((16,32,32),-1000.,dtype=np.float32)
synthetic[3:14,5:27,5:27] = 40.
mask,qc=ncct_head_tissue_support(synthetic)
assert mask.sum()==11*22*22
assert qc['clinical_brain_mask_validated'] is False
print('Patient-free support QC smoke: PASS', qc['coverage_fraction'])


## 3. Run a bounded fold-2 pilot without training

The script fits candidate parameters **only on six pilot training IDs**, writes local scratch NIfTI inputs for a **training-only** Dataset506 and Dataset507, and stages the held-out case in `imagesTs`. It never opens held-out labels. Patient-level QC results and calibrated parameters stay in your **private Google Drive**. The generated images remain under `/content/` and vanish when the runtime is reset.


In [ ]:
REPORTS.mkdir(parents=True,exist_ok=True)
SCRATCH=Path(tempfile.mkdtemp(prefix='wu20_fold2_pilot_',dir='/content'))
COMMAND=[sys.executable,'-u',str(SOURCE/'scripts'/'stage_isles2024_fold_pilot.py'),
    '--raw505',str(RAW),'--split-file',str(SPLIT),
    '--fold','2','--pilot-train-count','6','--pilot-validation-case','sub_stroke0016',
    '--scratch',str(SCRATCH),'--private-results',str(REPORTS)]
env=os.environ.copy()
env['PYTHONPATH']=str(SOURCE/'src')+os.pathsep+env.get('PYTHONPATH','')
print('Starting CPU-only pilot; transformed NIfTIs staged locally:',SCRATCH,flush=True)
logfile=REPORTS/'WU20_fold2_CPU_pilot.log'
with logfile.open('a',encoding='utf8') as log:
    with subprocess.Popen(COMMAND,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        rc=proc.wait()
if rc:raise RuntimeError(f'WU20 pilot failed (exit code {rc}). Check {logfile}')
result=json.loads((REPORTS/'WU20_pilot_fold2_aggregate.json').read_text())
assert result['training_ready'] is False
assert result['staging_inventory']['506']=={'n_train_images':6,'n_val_imagesTs':1,'n_train_labels':6}
assert result['staging_inventory']['507']=={'n_train_images':30,'n_val_imagesTs':5,'n_train_labels':6}
assert result['same_ncct_bytes_in_both_arms'] is True
print('Pilot staged and validated: PASS. Training authorization: BLOCKED.')


## 4. Inspect and preserve QC and exact next release gates

The full fold-specific normalization and 3d_fullres plan-lock integration are *not* certified by this small pilot. A failed mask/affine/finite check is an actionable blocker; do not lower thresholds to force a pass without documented independent review.


In [ ]:
from pprint import pprint
report=json.loads((REPORTS/'WU20_pilot_fold2_aggregate.json').read_text())
print('PILOT INVENTORY:')
pprint(report['staging_inventory'])
print('RELEASE BLOCKERS:')
for k in report['blocked_release_gates']:print(' -',k)
print('Private patient-level manifest:',REPORTS/'WU20_pilot_fold2_PRIVATE.json')
print('Aggregate:',REPORTS/'WU20_pilot_fold2_aggregate.json')
print('Local transformed images (NOT uploaded):',SCRATCH)
